<a href="https://colab.research.google.com/github/somustafa/areyoujoking/blob/main/joke.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate -U
!pip install deep-translator openai-whisper
!pip install evaluate

import os
import random
import torch
import numpy as np
import pandas as pd
import evaluate
import torch.nn.functional as F
import whisper
import shutil
from datasets import Dataset, load_dataset
from transformers import (
    BertTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
#dataseti hazirlayiriq
df = pd.read_csv('shortjokes.csv', engine='python')

pos_df = df[['Joke']].rename(columns={'Joke': 'text'})
pos_df['label'] = 1
pos_sample = pos_df.sample(min(20000, len(pos_df)), random_state=42)

def shuffle_text(text):
    words = str(text).split()
    random.shuffle(words)
    return " ".join(words)

shuffled_jokes = [shuffle_text(t) for t in pos_sample['text'].values]
neg_df_shuffled = pd.DataFrame({'text': shuffled_jokes, 'label': 0})

print("Wikipedia datasetindən ciddi cümlələr yüklənir...")
wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
real_sentences = [t for t in wiki['text'] if len(t) > 45][:20000]
neg_df_wiki = pd.DataFrame({'text': real_sentences, 'label': 0})

final_df = pd.concat([pos_sample, neg_df_shuffled, neg_df_wiki]).sample(frac=1).reset_index(drop=True)

print("\nYeni Dataset Balansı:")
print(final_df['label'].value_counts())

In [ ]:
# tokenization and dataset split
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

hg_dataset = Dataset.from_pandas(final_df)
tokenized_dataset = hg_dataset.map(preprocess_function, batched=True)
full_dataset = tokenized_dataset.train_test_split(test_size=0.2)

In [ ]:
# metrics ve modelin teyini
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hesablama cihazı: {device.upper()}")

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-multilingual-cased',
    num_labels=2,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)

print(f"Model Parametrləri: {model.num_parameters()}")

In [ ]:
# telim
training_args = TrainingArguments(
    output_dir="./joke_model_v2_serious",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.save_model("./joke_model_v2_serious")
tokenizer.save_pretrained("./joke_model_v2_serious")
shutil.make_archive('joke_model', 'zip', 'joke_model_v2_serious')

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
shutil.copy('joke_model.zip', '/content/drive/MyDrive/joke_model.zip')

Mounted at /content/drive
Fayl Google Drive-a köçürüldü! İndi Drive-dan rahatca endirə bilərsən.
